> **Notebook-first lesson.** Examples are executable. Download-dependent examples are guarded so the notebook can still run offline.

## Mathematical Framework

Math companions for this lesson:

- [Math 00 · Notation & Shapes](../../math/00_notation_shapes.ipynb)
- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 11 · Attention & Transformer Mathematics](../../math/11_attention_transformers.ipynb)

Do not stop at the API surface. Identify the **spaces/vectors involved, objective or probability model, local derivatives, matrix shapes, and approximation assumptions**.

# Lesson 43: Fine-tuning and LoRA

## Full fine-tuning
Update most or all pretrained model parameters for your task.

Benefits:
- maximum flexibility

Costs:
- memory
- compute
- storage
- risk of overfitting or catastrophic forgetting

## Parameter-efficient fine-tuning
LoRA introduces small low-rank trainable updates while leaving the main pretrained weights frozen.

Conceptually:

W' = W + BA

where B and A are low-rank matrices.

## Why low rank?
A rank-r update uses many fewer trainable parameters than an unrestricted full-size weight update.

## Experiment
Take a small pretrained Transformer and compare:
- frozen embeddings/features + classifier
- full fine-tuning
- LoRA/PEFT

Report trainable parameter count, memory use, training time and validation quality.

## Important
Fine-tuning does not guarantee factual correctness. Evaluation must match the intended task.


## Runnable activity
This is a reduced-scale experiment for the core mechanism. Run it first, then extend it.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
import torch
from torch import nn
class LoRALinear(nn.Module):
    def __init__(self,d_in,d_out,r=2):
        super().__init__()
        self.base=nn.Linear(d_in,d_out)
        for p in self.base.parameters(): p.requires_grad=False
        self.A=nn.Parameter(torch.randn(r,d_in)*.01)
        self.B=nn.Parameter(torch.zeros(d_out,r))
    def forward(self,x):
        return self.base(x)+(x@self.A.T)@self.B.T
for r in [1,2,4,8]:
    m=LoRALinear(64,64,r)
    trainable=sum(p.numel() for p in m.parameters() if p.requires_grad)
    full=64*64+64
    print("rank",r,"trainable",trainable,"full linear params",full,"ratio",trainable/full)

## Explanation checkpoint
Explain the mechanism, the scale gap between this activity and production/research systems, and one experiment you would run next.